# Generate paper tables and figures

Loads `results/*.csv` and writes lightweight reviewer-facing outputs to `figures/paper_figures/`. This notebook does not require raw datasets or checkpoints.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

RESULTS = Path("../results")
OUT = Path("../figures/paper_figures")
OUT.mkdir(parents=True, exist_ok=True)

files = sorted(RESULTS.glob("table_*.csv"))
if not files:
    raise FileNotFoundError("No result CSV files found in ../results")

tables = {path.stem: pd.read_csv(path) for path in files}
print("Loaded", len(tables), "CSV files")
print("\n".join(tables))

In [ ]:
summary_path = OUT / "manuscript_table_inventory.csv"
inventory = pd.DataFrame({
    "table_file": list(tables.keys()),
    "rows": [len(df) for df in tables.values()],
    "columns": [", ".join(df.columns) for df in tables.values()],
})
inventory.to_csv(summary_path, index=False)
inventory

In [ ]:
perf = tables["table_05_classification_performance"]
fig, ax = plt.subplots(figsize=(9, 4.8))
for dataset, sub in perf.groupby("dataset"):
    ax.plot(sub["model"], sub["f1_macro_mean"].astype(float), marker="o", label=dataset)
ax.set_ylabel("F1-macro")
ax.set_xlabel("Model")
ax.set_title("Classification performance by dataset")
ax.grid(axis="y", alpha=0.3)
ax.legend()
fig.tight_layout()
fig.savefig(OUT / "table_05_classification_performance.png", dpi=180)
plt.show()

In [ ]:
rq2 = tables["table_08_rq2_model_variation_pairwise"]
fig, ax = plt.subplots(figsize=(9, 4.8))
for dataset, sub in rq2.groupby("dataset"):
    ax.plot(sub["model"], sub["spearman_mean"].astype(float), marker="o", label=dataset)
ax.set_ylabel("Pairwise SHAP Spearman")
ax.set_xlabel("Model")
ax.set_title("RQ2 model-variation consistency")
ax.grid(axis="y", alpha=0.3)
ax.legend()
fig.tight_layout()
fig.savefig(OUT / "table_08_rq2_pairwise_spearman.png", dpi=180)
plt.show()